# WeightedKgBlend — Ensemble (slice_0)
Combines TransE + RotatE + ProbCBR predictions using Optuna-optimized weights.

**Scoring:** For each drug, collect top-50 candidate diseases from all models.
Score each candidate: `Σ λᵢ × (1/rank_in_model_i)` where rank is position in top-50 (0 if not present).
Re-rank candidates by combined score → find rank of expected disease → MRR.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from collections import defaultdict

RESULTS = Path('/Users/meghamala/projects/WeightedKgBlend/results')
SLICE   = 'slice_0'
TOP_K   = 50

MODELS = ['TransE', 'RotatE', 'ProbCBR']

# Load predictions
preds = {}
for model in MODELS:
    for split in ['valid', 'test']:
        path = RESULTS / model / SLICE / f'predictions_{split}.tsv'
        preds[(model, split)] = pd.read_csv(path, sep='\t')
        print(f'{model:10s} {split:5s}: {len(preds[(model, split)]):,} rows')

print('\nAll loaded.')

In [ ]:
# Individual MRRs
print('Individual MRRs (slice_0):')
print(f'{"Model":12s} {"Valid MRR":>10s} {"Test MRR":>10s}')
print('-' * 35)
for model in MODELS:
    v_mrr = preds[(model, 'valid')]['reciprocal_rank'].mean()
    t_mrr = preds[(model, 'test')]['reciprocal_rank'].mean()
    print(f'{model:12s} {v_mrr:>10.4f} {t_mrr:>10.4f}')

In [ ]:
def compute_ensemble_mrr(dfs, weights, top_k=TOP_K):
    """
    dfs: list of DataFrames (one per model), aligned on same rows
    weights: list of floats, one per model
    
    For each drug:
      - Collect all candidate diseases from top-k columns of all models
      - Score: sum of λi × (1/position) for each model
      - Re-rank → find rank of expected disease → RR
    """
    rrs = []
    n_rows = len(dfs[0])

    for idx in range(n_rows):
        exp_dis = dfs[0].iloc[idx]['expected_disease']
        candidate_scores = defaultdict(float)

        for model_idx, df in enumerate(dfs):
            row = df.iloc[idx]
            for k in range(1, top_k + 1):
                disease = row.get(f'top{k}_disease', '')
                if disease:
                    candidate_scores[disease] += weights[model_idx] * (1.0 / k)

        ranked = sorted(candidate_scores, key=candidate_scores.get, reverse=True)
        rank   = ranked.index(exp_dis) + 1 if exp_dis in ranked else len(ranked) + 1
        rrs.append(1.0 / rank)

    return float(np.mean(rrs))


def align_dfs(split):
    """Merge all model predictions on (drug, expected_disease) to ensure alignment."""
    base = preds[(MODELS[0], split)][['drug', 'expected_disease']].copy()
    aligned = []
    for model in MODELS:
        df = preds[(model, split)]
        merged = base.merge(df, on=['drug', 'expected_disease'], how='left')
        aligned.append(merged)
    return aligned

valid_dfs = align_dfs('valid')
test_dfs  = align_dfs('test')

# Sanity check with equal weights
equal_weights = [1/3, 1/3, 1/3]
print(f'Equal-weight ensemble valid MRR: {compute_ensemble_mrr(valid_dfs, equal_weights):.4f}')
print(f'Equal-weight ensemble test  MRR: {compute_ensemble_mrr(test_dfs,  equal_weights):.4f}')

In [ ]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    # Sample weights and normalize so they sum to 1
    w = [trial.suggest_float(f'w{i}', 0.0, 1.0) for i in range(len(MODELS))]
    total = sum(w) or 1.0
    w = [wi / total for wi in w]
    return compute_ensemble_mrr(valid_dfs, w)

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=200, show_progress_bar=True)

best_w_raw = [study.best_params[f'w{i}'] for i in range(len(MODELS))]
total = sum(best_w_raw)
best_weights = [w / total for w in best_w_raw]

print('\nOptimal weights:')
for model, w in zip(MODELS, best_weights):
    print(f'  {model:12s}: {w:.4f}')
print(f'\nBest valid MRR : {study.best_value:.4f}')

In [ ]:
# Final evaluation on test set with optimal weights
ensemble_test_mrr = compute_ensemble_mrr(test_dfs, best_weights)

print('=' * 40)
print('Final Results — slice_0')
print('=' * 40)
print(f'{"Model":12s} {"Valid MRR":>10s} {"Test MRR":>10s}')
print('-' * 35)
for model in MODELS:
    v = preds[(model, 'valid')]['reciprocal_rank'].mean()
    t = preds[(model, 'test')]['reciprocal_rank'].mean()
    print(f'{model:12s} {v:>10.4f} {t:>10.4f}')
print('-' * 35)
print(f'{"Ensemble":12s} {study.best_value:>10.4f} {ensemble_test_mrr:>10.4f}')
print('=' * 40)
print(f'\nWeights: ' + ', '.join(f'{m}={w:.3f}' for m, w in zip(MODELS, best_weights)))